<a href="https://colab.research.google.com/github/danilo576/Cuda/blob/main/CUDA_Paralelni_sistemi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CUDA — Paralelni sistemi

Materijal i vežbe iz programiranja na GPU-u pomoću NVIDIA CUDA tehnologije.

# ⚙️ 1. Podešavanje okruženja

In [1]:
!pip install -q nvcc4jupyter

In [2]:
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpovmycf36".


## 1.1 Provera GPU-a

## 1.2 Provera CUDA kompajlera

In [4]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


# 🚀 2. Prvi CUDA program

Pre nego što krenemo sa vežbama, napravićemo jednostavan CUDA program koji izvršava funkciju na GPU-u.

In [5]:
%%cuda

#include <stdio.h>

__global__ void hello() {
    printf("Hello from GPU!\n");
}

int main() {
    hello<<<1, 1>>>();
    cudaDeviceSynchronize();

    return 0;
}

Hello from GPU!



## 2.1 CPU i GPU

In [6]:
%%cuda

#include <stdio.h>

__global__ void hello_from_gpu() {
    printf("Hello from GPU!\n");
}

int main() {

    printf("Hello from CPU!\n");

    hello_from_gpu<<<1, 1>>>();

    cudaDeviceSynchronize();

    return 0;
}

Hello from CPU!
Hello from GPU!



## 2.2 Blokovi i threadovi

In [7]:
%%cuda

#include <stdio.h>

__global__ void hello() {

    printf(
        "Block: %d, Thread: %d\n",
        blockIdx.x,
        threadIdx.x
    );
}

int main() {

    hello<<<2, 4>>>();

    cudaDeviceSynchronize();

    return 0;
}

Block: 1, Thread: 0
Block: 1, Thread: 1
Block: 1, Thread: 2
Block: 1, Thread: 3
Block: 0, Thread: 0
Block: 0, Thread: 1
Block: 0, Thread: 2
Block: 0, Thread: 3



# 🧮 3. Sabiranje vektora

Sabiranje dva vektora korišćenjem CUDA kernela.

Za svaki element važi:
$
C[i] = A[i] + B[i]
$

In [41]:
%%cuda

#include <stdio.h>

#define N 10

__global__ void vector_add(const int *A, const int *B, int *C) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < N) {
        C[i] = A[i] + B[i];
    }
}

int main() {

    int A[N], B[N], C[N];

    for (int i = 0; i < N; i++) {
        A[i] = i;
        B[i] = i * 2;
    }

    int *d_A, *d_B, *d_C;

    cudaMalloc(&d_A, N * sizeof(int));
    cudaMalloc(&d_B, N * sizeof(int));
    cudaMalloc(&d_C, N * sizeof(int));

    cudaMemcpy(d_A, A, N * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, N * sizeof(int), cudaMemcpyHostToDevice);

    vector_add<<<1, N>>>(d_A, d_B, d_C);

    cudaDeviceSynchronize();

    cudaMemcpy(C, d_C, N * sizeof(int), cudaMemcpyDeviceToHost);

    for (int i = 0; i < N; i++) {
        printf("C[%d] = %d\n", i, C[i]);
    }

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

C[0] = 0
C[1] = 3
C[2] = 6
C[3] = 9
C[4] = 12
C[5] = 15
C[6] = 18
C[7] = 21
C[8] = 24
C[9] = 27



# 📝 Zadatak 1 — Operacija nad vektorima

Napisati CUDA program koji računa izraz:

$
C = A * x + B
$

gde su:

- `A` i `B` — vektori,
- `x` — skalar,
- `C` — rezultujući vektor.

Za svaki element važi:

$
C[i] = A[i] * x + B[i]
$

Potrebno je:

1. Izračunati rezultat paralelno korišćenjem CUDA kernela.
2. Izračunati isti rezultat sekvencijalno na CPU-u.
3. Uporediti rezultate i proveriti validnost CUDA rešenja.
4. Analizirati ponašanje programa pri promeni:
   - broja blokova,
   - broja niti u jednom bloku.

U CUDA kernelu koristi se globalni indeks niti:

```cpp
int tid = blockIdx.x * blockDim.x + threadIdx.x;

In [16]:
%%cuda

#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define N 5120
#define BLOCK_SIZE 32

__host__ __device__ int operation(int a, int b, int x)
{
    return a * x + b;
}

__global__ void operate(int *A, int *B, int *C, int x)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    for (int i = tid; i < N; i += gridDim.x * blockDim.x)
    {
        C[i] = operation(A[i], B[i], x);
    }
}

__host__ bool check_result(int *A, int *B, int *GPU_C, int x)
{
    for (int i = 0; i < N; i++)
    {
        int CPU_C = operation(A[i], B[i], x);

        if (GPU_C[i] != CPU_C)
        {
            return false;
        }
    }

    return true;
}

__host__ void operate_on_GPU(int *A, int *B, int *C, int x)
{
    int *dev_A;
    int *dev_B;
    int *dev_C;

    cudaMalloc((void **)&dev_A, N * sizeof(int));
    cudaMalloc((void **)&dev_B, N * sizeof(int));
    cudaMalloc((void **)&dev_C, N * sizeof(int));

    cudaMemcpy(dev_A, A, N * sizeof(int), cudaMemcpyHostToDevice);

    cudaMemcpy(
        dev_B,
        B,
        N * sizeof(int),
        cudaMemcpyHostToDevice
    );

    dim3 blockSize(BLOCK_SIZE);
    dim3 gridSize((N + BLOCK_SIZE - 1) / BLOCK_SIZE);

    operate<<<gridSize, blockSize>>>(dev_A, dev_B, dev_C, x);

    cudaDeviceSynchronize();

    cudaMemcpy(C, dev_C, N * sizeof(int), cudaMemcpyDeviceToHost);

    cudaFree(dev_A);
    cudaFree(dev_B);
    cudaFree(dev_C);
}

int main()
{
    int *A;
    int *B;
    int *GPU_C;

    int x = 5;

    A = (int *)malloc(N * sizeof(int));
    B = (int *)malloc(N * sizeof(int));
    GPU_C = (int *)malloc(N * sizeof(int));

    for (int i = 0; i < N; i++)
    {
        A[i] = i + 1;
        B[i] = N - (i + 1);
    }

    operate_on_GPU(A, B, GPU_C, x);

    printf("Prvih 10 elemenata vektora C:\n\n");

    for (int i = 0; i < 10; i++)
    {
        printf("C[%d] = %d\n", i, GPU_C[i]);
    }

    if (check_result(A, B, GPU_C, x))
    {
        printf("\nRezultat je tacan!\n");
    }
    else
    {
        printf("\nRezultat je netacan!\n");
    }

    free(A);
    free(B);
    free(GPU_C);

    return 0;
}

Prvih 10 elemenata vektora C:

C[0] = 5124
C[1] = 5128
C[2] = 5132
C[3] = 5136
C[4] = 5140
C[5] = 5144
C[6] = 5148
C[7] = 5152
C[8] = 5156
C[9] = 5160

Rezultat je tacan!



# 📝 Zadatak 2 — Sabiranje dve matrice

Napisati CUDA program koji računa izraz:

$
C = A + B
$

gde su:

- `A` i `B` — kvadratne matrice dimenzija `N × N`,
- `C` — rezultujuća matrica.

Za svaki element važi:

$
C[i][j] = A[i][j] + B[i][j]
$

Potrebno je:

1. Izračunati rezultat paralelno korišćenjem CUDA kernela.
2. Izračunati isti rezultat sekvencijalno na CPU-u.
3. Uporediti rezultate i proveriti validnost CUDA rešenja.
4. Analizirati ponašanje programa pri promeni:
   - broja blokova,
   - broja niti u okviru jednog bloka.

Pošto obrađujemo matricu, koristimo dvodimenzionalne blokove i grid.

Globalni indeks vrste i kolone:

```cpp
int row = blockIdx.y * blockDim.y + threadIdx.y;
int col = blockIdx.x * blockDim.x + threadIdx.x;


In [23]:
%%cuda

#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define N 512
#define BLOCK_SIZE 32

__host__ void operate_on_GPU(int *A, int *B, int *C);
__global__ void operate(int *A, int *B, int *C);
__host__ bool check_result(int *A, int *B, int *GPU_C);

__device__ __host__ int operation(int a, int b)
{
    return a + b;
}

int main()
{
    int *A, *B, *GPU_C;

    A = (int *)malloc(sizeof(int) * N * N);
    B = (int *)malloc(sizeof(int) * N * N);
    GPU_C = (int *)malloc(sizeof(int) * N * N);

    for (int i = 0; i < N; i++)
    {
        for (int j = 0; j < N; j++)
        {
            A[i * N + j] = i * N + j + 1;
            B[i * N + j] = j * N + i + 1;
        }
    }

    operate_on_GPU(A, B, GPU_C);

    printf("Prvih 8x8 elemenata matrice C:\n\n");

    for (int i = 0; i < 8; i++)
    {
        printf("|\t");

        for (int j = 0; j < 8; j++)
            printf("%d\t", GPU_C[i * N + j]);

        printf("|\n");
    }

    if (check_result(A, B, GPU_C))
        printf("\nRezultat je tacan!\n");
    else
        printf("\nRezultat je netacan!\n");

    free(GPU_C);
    free(B);
    free(A);

    return 0;
}

__host__ bool check_result(int *A, int *B, int *GPU_C)
{
    for (int i = 0; i < N; i++)
    {
        for (int j = 0; j < N; j++)
        {
            int index = i * N + j;

            if (GPU_C[index] != operation(A[index], B[index]))
                return false;
        }
    }

    return true;
}

__host__ void operate_on_GPU(int *A, int *B, int *C)
{
    int *dev_A, *dev_B, *dev_C;
    size_t size = sizeof(int) * N * N;

    cudaMalloc((void **)&dev_A, size);
    cudaMalloc((void **)&dev_B, size);
    cudaMalloc((void **)&dev_C, size);

    cudaMemcpy(dev_A, A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(dev_B, B, size, cudaMemcpyHostToDevice);

    dim3 blockSize(BLOCK_SIZE, BLOCK_SIZE);

    dim3 gridSize(
        (N + blockSize.x - 1) / blockSize.x,
        (N + blockSize.y - 1) / blockSize.y
    );

    operate<<<gridSize, blockSize>>>(dev_A, dev_B, dev_C);

    cudaDeviceSynchronize();

    cudaMemcpy(C, dev_C, size, cudaMemcpyDeviceToHost);

    cudaFree(dev_A);
    cudaFree(dev_B);
    cudaFree(dev_C);
}

__global__ void operate(int *A, int *B, int *C)
{
    int tid_x = blockIdx.x * blockDim.x + threadIdx.x;
    int tid_y = blockIdx.y * blockDim.y + threadIdx.y;

    for (int i = tid_x; i < N; i += gridDim.x * blockDim.x)
    {
        for (int j = tid_y; j < N; j += gridDim.y * blockDim.y)
        {
            C[i * N + j] = operation(A[i * N + j], B[i * N + j]);
        }
    }
}

Prvih 8x8 elemenata matrice C:

|	2	515	1028	1541	2054	2567	3080	3593	|
|	515	1028	1541	2054	2567	3080	3593	4106	|
|	1028	1541	2054	2567	3080	3593	4106	4619	|
|	1541	2054	2567	3080	3593	4106	4619	5132	|
|	2054	2567	3080	3593	4106	4619	5132	5645	|
|	2567	3080	3593	4106	4619	5132	5645	6158	|
|	3080	3593	4106	4619	5132	5645	6158	6671	|
|	3593	4106	4619	5132	5645	6158	6671	7184	|

Rezultat je tacan!



# 📝 Zadatak 3 — Sabiranje vektora sa zamenom elemenata

Napisati CUDA program koji računa izraz:

$
C = A + B
$

gde su:

- `A` i `B` — vektori,
- `C` — rezultujući vektor,
- `k` — granična vrednost,
- `p` — vrednost kojom se vrši zamena.

Za svaki element najpre računamo:

$
C[i] = A[i] + B[i]
$

Ako je zbir veći od `k`, rezultat se zamenjuje vrednošću `p`:

$
C[i] =
\begin{cases}
p, & \text{ako } A[i] + B[i] > k \\
A[i] + B[i], & \text{ako } A[i] + B[i] \leq k
\end{cases}
$

Potrebno je:

1. Izračunati rezultat paralelno korišćenjem CUDA kernela.
2. Omogućiti unos vrednosti `k` i `p`.
3. Izračunati isti rezultat sekvencijalno na CPU-u.
4. Uporediti CPU i GPU rezultat.
5. Analizirati ponašanje programa pri promeni broja blokova i broja niti po bloku.

Globalni indeks niti računa se kao:

```cpp
int tid = blockIdx.x * blockDim.x + threadIdx.x;
```

Za obradu vektora koristi se grid-stride petlja:

```cpp
for (int i = tid; i < N; i += gridDim.x * blockDim.x)
```

In [38]:
%%writefile zadatak5.cu

#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define N 5120
#define BLOCK_SIZE 32

__host__ void operate_on_GPU(int *A, int *B, int *C, int k, int p);
__global__ void operate(int *A, int *B, int *C, int k, int p);
__host__ bool check_result(int *A, int *B, int *GPU_C, int k, int p);

__device__ __host__ int operation(int a, int b, int k, int p)
{
    int result = a + b;
    return result > k ? p : result;
}

int main(int argc, char **argv)
{
    int *A, *B, *GPU_C;
    int k, p;

    if (argc != 3)
    {
        printf("Unesite k i p.\n");
        return 1;
    }

    k = atoi(argv[1]);
    p = atoi(argv[2]);

    A = (int *)malloc(sizeof(int) * N);
    B = (int *)malloc(sizeof(int) * N);
    GPU_C = (int *)malloc(sizeof(int) * N);

    for (int i = 0; i < N; i++)
    {
        A[i] = i * i;
        B[i] = N - (i + 1);
    }

    operate_on_GPU(A, B, GPU_C, k, p);

    printf("k = %d, p = %d\n\n", k, p);
    printf("Prvih 20 elemenata vektora C:\n");

    for (int i = 0; i < 20; i++)
        printf("C[%d] = %d\n", i, GPU_C[i]);

    if (check_result(A, B, GPU_C, k, p))
        printf("\nRezultat je tacan!\n");
    else
        printf("\nRezultat je netacan!\n");

    free(GPU_C);
    free(B);
    free(A);

    return 0;
}

__host__ bool check_result(int *A, int *B, int *GPU_C, int k, int p)
{
    for (int i = 0; i < N; i++)
    {
        if (GPU_C[i] != operation(A[i], B[i], k, p))
            return false;
    }

    return true;
}

__host__ void operate_on_GPU(int *A, int *B, int *C, int k, int p)
{
    int *dev_A, *dev_B, *dev_C;
    size_t size = sizeof(int) * N;

    cudaMalloc((void **)&dev_A, size);
    cudaMalloc((void **)&dev_B, size);
    cudaMalloc((void **)&dev_C, size);

    cudaMemcpy(dev_A, A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(dev_B, B, size, cudaMemcpyHostToDevice);

    dim3 blockSize(BLOCK_SIZE);
    dim3 gridSize((N + blockSize.x - 1) / blockSize.x);

    operate<<<gridSize, blockSize>>>(dev_A, dev_B, dev_C, k, p);

    cudaDeviceSynchronize();

    cudaMemcpy(C, dev_C, size, cudaMemcpyDeviceToHost);

    cudaFree(dev_A);
    cudaFree(dev_B);
    cudaFree(dev_C);
}

__global__ void operate(int *A, int *B, int *C, int k, int p)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    for (int i = tid; i < N; i += gridDim.x * blockDim.x)
        C[i] = operation(A[i], B[i], k, p);
}

Overwriting zadatak5.cu


In [39]:
!nvcc zadatak5.cu -o zadatak5

In [40]:
k = 5200 #@param {type:"integer"}
p = 100 #@param {type:"integer"}

!./zadatak5 {k} {p}

k = 5200, p = 100

Prvih 20 elemenata vektora C:
C[0] = 5119
C[1] = 5119
C[2] = 5121
C[3] = 5125
C[4] = 5131
C[5] = 5139
C[6] = 5149
C[7] = 5161
C[8] = 5175
C[9] = 5191
C[10] = 100
C[11] = 100
C[12] = 100
C[13] = 100
C[14] = 100
C[15] = 100
C[16] = 100
C[17] = 100
C[18] = 100
C[19] = 100

Rezultat je tacan!
